# Stage 0b — Supervised-Contrastive DINOv2 Fine-Tuning

Thin notebook: it only **imports**, **calls** `src/`, and **displays**. All
pair-sampling / NT-Xent / training-loop logic lives in
`src/contrastive_finetune.py`.

Unfreezes only the last `contrastive_finetune.unfreeze_last_n_blocks`
transformer blocks of `analysis.model` (frozen everywhere else) and trains
with supervised NT-Xent: positive pairs are two architectures' main figures
sharing `contrastive_finetune.class_col` (e.g. `G1_topType`) but differing on
`contrastive_finetune.style_cols` (e.g. `acSty`) — pulling same-class,
different-style figures together so the embedding space stops separating by
drawing style.

This can run unattended (e.g. overnight): progress and GPU stats are logged
to a file under `contrastive_finetune.output_dir/logs/`, not just stdout.

Fill in the `EDIT-ME` values in `config.yaml`
(`paths.embeddings_root`, `contrastive_finetune.output_dir`) before running.

**After this notebook finishes**, open repo 3's (`eVTOL-Embedding-Evaluation`)
`20_probes.ipynb` / `21_structure_clustering.ipynb` and pick
`contrastive_finetune.output_pipeline_name` (default `dinov2_contrastive_ft`)
as the pipeline via `registry.py`, alongside `dinov2_frozen`, to compare
before/after directly — that diagnostic ladder is repo 3's code and stays
there; this notebook does not reimplement or import it.

In [ ]:
import sys
from pathlib import Path

# Locate the folder root (the dir containing config.yaml) and make src importable.
ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src import contrastive_finetune as cft

cfg = load_config()
fcfg = cfg['contrastive_finetune']
print('base model      :', cfg['analysis']['model'])
print('class_col       :', fcfg['class_col'])
print('style_cols      :', fcfg['style_cols'])
print('unfreeze blocks :', fcfg['unfreeze_last_n_blocks'])
print('device          :', fcfg['device'])
print('lr / batch / T  :', fcfg['learning_rate'], '/', fcfg['batch_size'], '/', fcfg['temperature'])
print('num_epochs      :', fcfg['num_epochs'])
print('output_dir      :', fcfg['output_dir'])
print('embeddings_root :', cfg['paths']['embeddings_root'])
print('output pipeline :', fcfg['output_pipeline_name'])

In [ ]:
# Preview the class/style-eligible pair source before committing to a full run.
pair_table = cft.load_pair_table(cfg)
pair_table[[fcfg['class_col']] + fcfg['style_cols']].value_counts().head(20)

In [ ]:
# Train. Logs epoch/loss + GPU stats to contrastive_finetune.output_dir/logs/
# and checkpoints to contrastive_finetune.output_dir/checkpoints/ as it goes,
# so an unattended run can be inspected or resumed-from-checkpoint mid-flight.
result = cft.run_finetune(cfg)
result['history']

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(result['history']['epoch'], result['history']['avg_loss'], marker='o')
plt.xlabel('epoch'); plt.ylabel('avg NT-Xent loss')
plt.title('Contrastive fine-tune loss')
plt.tight_layout(); plt.show()

In [ ]:
# Extract embeddings with the fine-tuned model and write them as a new
# pipeline folder under paths.embeddings_root, in repo 3's registry.py
# format (emb_*.npy + metadata.parquet + manifest.json) — same folder that
# already holds the frozen-baseline pipeline(s), so repo 3 can compare them
# without any code change on its side.
pipeline_dir = cft.save_finetuned_pipeline(result, cfg)
pipeline_dir